In [8]:
from pathlib import Path
import pandas as pd
import nltk

# =========================
# 1. Tìm project root
# =========================
relative_path = Path(
    "data/processed/complaints_clean.pkl"
)

project_root = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / relative_path).is_file()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError(
        f"Could not find {relative_path} from {Path.cwd()}."
    )

# =========================
# 2. Download NLTK resources
# =========================
resources = [
    ("corpora/stopwords", "stopwords"),
    ("corpora/wordnet", "wordnet"),
    ("corpora/omw-1.4", "omw-1.4"),
]

for path, resource in resources:
    try:
        nltk.data.find(path)
    except LookupError:
        nltk.download(resource)

# =========================
# 3. Load cleaned complaints
# =========================
complaints_clean = pd.read_pickle(
    project_root / relative_path
)

print("Project root:", project_root)
print("Shape:", complaints_clean.shape)
print()
print("Columns:")
print(complaints_clean.columns.tolist())

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\T460P\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\T460P\AppData\Roaming\nltk_data...
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\T460P\AppData\Roaming\nltk_data...


Project root: d:\data mining\e_waste_topic_modeling
Shape: (130757, 13)

Columns:
['asin', 'reviewText', 'summary', 'overall', 'reviewTime', 'unixReviewTime', 'verified', 'vote', 'review_date', 'year', 'text_length', 'text_for_nlp', 'summary_for_nlp']


In [9]:
# Lấy một số review đại diện để kiểm tra preprocessing
sample_reviews = complaints_clean[
    ["reviewText", "summary"]
].sample(
    n=20,
    random_state=42
)

display(sample_reviews)

,reviewText,summary
68361,Bought to use external hard drive on the Xoom ...,Nonworking on Xoom 4.11
185156,"It looks great. It feels great, but the rubbe...",I wish the case did not fall apart.
865151,"The charge rate is specified as being ""up to 4...",Charge rate is significantly lower than the sp...
520046,The charger doesn't charge the phone,One Star
738233,"It could be a good thing, but it has a lot of ...",Could be awesome..... but isn't
212217,Wish I could give zero stars. Ordered this wit...,DO NOT BUY
1005739,"I could not use the controls on the watch, pre...","held moisture, could not swipe watch"
1076378,When buying screen protectors and they come in...,Absolute garbage!
59811,I buy eforcity products all the time for quali...,Dropped the ball twice
245594,ive had this thing not even a week and it alre...,dont buy it.


In [10]:
# Một số review có emoticon / contraction / ký tự đặc biệt
patterns = r"(:\)|:\(|:-\)|:-\(|doesn't|don't|didn't|can't|won't|battery life|charging port|touch screen)"

special_reviews = complaints_clean[
    complaints_clean["reviewText"]
    .str.contains(
        patterns,
        case=False,
        regex=True,
        na=False
    )
][["reviewText", "summary"]]

display(
    special_reviews.sample(
        n=min(30, len(special_reviews)),
        random_state=42
    )
)

C:\Users\T460P\AppData\Local\Temp\ipykernel_6932\2213748172.py:6: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  .str.contains(


,reviewText,summary
979116,I don't know what phone this fits but it is no...,Does not fit at all.
487373,Purchased this for my Samsung galaxy and it di...,Purchased this for my Samsung galaxy and it di...
511210,"I'm really disappointed, it didn't even fit my...","I'm really disappointed, it didn't even fit my..."
316578,Even though I have been using an Android table...,This smart phone makes me feel stupid.
797249,Beautiful case BUT ... there are 2 parts to it...,Beautiful case BUT
112244,I have installed I don't know how many screen ...,Worst screen covers ever
53173,This thing did not even mount in my car anywhe...,Stuck with trash
138905,Update: Warning! Dangerous! Burn risk!\n\nWell...,"Warning! ""Custom Unique Proprietary Charger!""..."
883088,I have the iOttie Easy One Touch 2 and use it ...,Flawed mechanism. Previous one was superior.
402877,"Didn't fit my phone, I guess it's not describ...","Didn't fit my phone, I guess it's not describe..."


In [11]:
import re
import html


def normalize_text(text):
    """
    Normalize review text while preserving useful linguistic signals.

    Keeps:
    - contractions: don't, doesn't, didn't, can't...
    - emoticons: :(, :), :-(, :-)
    - words and punctuation needed for later tokenization
    """

    if pd.isna(text):
        return ""

    text = str(text)

    # 1. Decode HTML entities
    # Example: &amp; -> &
    text = html.unescape(text)

    # 2. Remove HTML tags
    # Example: <div class="..."> -> ""
    text = re.sub(r"<[^>]+>", " ", text)

    # 3. Remove URLs
    text = re.sub(
        r"https?://\S+|www\.\S+",
        " ",
        text,
        flags=re.IGNORECASE
    )

    # 4. Remove escaped line breaks / tabs
    # Example: \n, \t appearing literally in text
    text = re.sub(r"\\[nrt]", " ", text)

    # 5. Normalize real whitespace
    text = re.sub(r"\s+", " ", text)

    # 6. Lowercase
    text = text.lower()

    # 7. Normalize repeated punctuation
    # Keep meaningful punctuation/emoticons,
    # but reduce excessive repetitions.
    text = re.sub(r"!{3,}", "!!", text)
    text = re.sub(r"\?{3,}", "??", text)
    text = re.sub(r"\.{4,}", "...", text)

    # 8. Final whitespace cleanup
    text = text.strip()

    return text

In [12]:
test_reviews = [
    "The charger doesn't charge the phone",
    "I was really disappointed, it didn't even fit my phone.",
    "Can't get fast charging. Always drops when traveling.",
    "I'm sad to say this does not hold an LG V20. :(",
    "Great product!!! But it doesn't work???",
    '<div id="video-block">This product is terrible</div>',
    r"Update: Warning! Dangerous! Burn risk:\n\nWell...",
    "THIS IS GARBAGE!!! DOES NOT FIT GALAXY S7 EDGE",
    "It looks great. It feels great, but the rubber fell apart.",
]

test_normalized = pd.DataFrame({
    "original": test_reviews,
    "normalized": [normalize_text(text) for text in test_reviews]
})

display(test_normalized)

,original,normalized
0,The charger doesn't charge the phone,the charger doesn't charge the phone
1,"I was really disappointed, it didn't even fit ...","i was really disappointed, it didn't even fit ..."
2,Can't get fast charging. Always drops when tra...,can't get fast charging. always drops when tra...
3,I'm sad to say this does not hold an LG V20. :(,i'm sad to say this does not hold an lg v20. :(
4,Great product!!! But it doesn't work???,great product!! but it doesn't work??
5,"<div id=""video-block"">This product is terrible...",this product is terrible
6,Update: Warning! Dangerous! Burn risk:\n\nWell...,update: warning! dangerous! burn risk: well...
7,THIS IS GARBAGE!!! DOES NOT FIT GALAXY S7 EDGE,this is garbage!! does not fit galaxy s7 edge
8,"It looks great. It feels great, but the rubber...","it looks great. it feels great, but the rubber..."


In [13]:
# Apply normalization to the full complaint corpus

complaints_clean["normalized_text"] = (
    complaints_clean["reviewText"]
    .map(normalize_text)
)

print("Number of reviews:", len(complaints_clean))
print("Missing normalized text:",
      complaints_clean["normalized_text"].isna().sum())

print()
display(
    complaints_clean[
        ["reviewText", "normalized_text"]
    ].head(10)
)

Number of reviews: 130757
Missing normalized text: 0



,reviewText,normalized_text
3,DON'T CARE FOR IT. GAVE IT AS A GIFT AND THEY...,don't care for it. gave it as a gift and they ...
5,The product looked exactly like the picture an...,the product looked exactly like the picture an...
8,DO NOT BUY! this item is seriously cheap as he...,do not buy! this item is seriously cheap as he...
12,Very cheap broke the first time we put it on :...,very cheap broke the first time we put it on :...
17,I used this case for not even a week and the b...,i used this case for not even a week and the b...
20,The bow fell off multiple times I had to keep ...,the bow fell off multiple times i had to keep ...
72,I was very excited when I first got this case....,i was very excited when i first got this case....
75,Sent case that didn't fit,sent case that didn't fit
76,The picture looks waaayyy better than the actu...,the picture looks waaayyy better than the actu...
83,I loved the case when I first received it but ...,i loved the case when i first received it but ...


In [15]:
import nltk

try:
    nltk.data.find("tokenizers/punkt")
except LookupError:
    nltk.download("punkt")

try:
    nltk.data.find("tokenizers/punkt_tab")
except LookupError:
    nltk.download("punkt_tab")

In [16]:
from nltk.tokenize import word_tokenize

def tokenize_text(text):
    return word_tokenize(text)

complaints_clean["tokens"] = (
    complaints_clean["normalized_text"]
    .map(tokenize_text)
)

print("Tokenization completed.")
print()

display(
    complaints_clean[
        ["normalized_text", "tokens"]
    ].head(10)
)

Tokenization completed.



,normalized_text,tokens
3,don't care for it. gave it as a gift and they ...,"[do, n't, care, for, it, ., gave, it, as, a, g..."
5,the product looked exactly like the picture an...,"[the, product, looked, exactly, like, the, pic..."
8,do not buy! this item is seriously cheap as he...,"[do, not, buy, !, this, item, is, seriously, c..."
12,very cheap broke the first time we put it on :...,"[very, cheap, broke, the, first, time, we, put..."
17,i used this case for not even a week and the b...,"[i, used, this, case, for, not, even, a, week,..."
20,the bow fell off multiple times i had to keep ...,"[the, bow, fell, off, multiple, times, i, had,..."
72,i was very excited when i first got this case....,"[i, was, very, excited, when, i, first, got, t..."
75,sent case that didn't fit,"[sent, case, that, did, n't, fit]"
76,the picture looks waaayyy better than the actu...,"[the, picture, looks, waaayyy, better, than, t..."
83,i loved the case when i first received it but ...,"[i, loved, the, case, when, i, first, received..."


In [45]:
def normalize_tokens(tokens):
    """
    Normalize tokenized text:
    - Expand common contractions
    - Convert emoticons into semantic tokens
    """

    normalized = []
    i = 0

    while i < len(tokens):
        token = tokens[i]

        # =====================================
        # 1. Sad emoticons
        # =====================================

        # :(
        if (
            token == ":"
            and i + 1 < len(tokens)
            and tokens[i + 1] == "("
        ):
            normalized.append("EMOTION_SAD")
            i += 2
            continue

        # :-(
        if (
            token == ":"
            and i + 2 < len(tokens)
            and tokens[i + 1] == "-"
            and tokens[i + 2] == "("
        ):
            normalized.append("EMOTION_SAD")
            i += 3
            continue

        # D:<
        if (
            token == "d"
            and i + 2 < len(tokens)
            and tokens[i + 1] == ":"
            and tokens[i + 2] == "<"
        ):
            normalized.append("EMOTION_SAD")
            i += 3
            continue

        # =====================================
        # 2. Happy emoticons
        # =====================================

        # :)
        if (
            token == ":"
            and i + 1 < len(tokens)
            and tokens[i + 1] == ")"
        ):
            normalized.append("EMOTION_HAPPY")
            i += 2
            continue

        # :-)
        if (
            token == ":"
            and i + 2 < len(tokens)
            and tokens[i + 1] == "-"
            and tokens[i + 2] == ")"
        ):
            normalized.append("EMOTION_HAPPY")
            i += 3
            continue

        # =====================================
        # 3. Negative contractions
        # =====================================

        # can't → can not
        if (
            token == "ca"
            and i + 1 < len(tokens)
            and tokens[i + 1] == "n't"
        ):
            normalized.extend(["can", "not"])
            i += 2
            continue

        # won't → will not
        if (
            token == "wo"
            and i + 1 < len(tokens)
            and tokens[i + 1] == "n't"
        ):
            normalized.extend(["will", "not"])
            i += 2
            continue

        # shan't → shall not
        if (
            token == "sha"
            and i + 1 < len(tokens)
            and tokens[i + 1] == "n't"
        ):
            normalized.extend(["shall", "not"])
            i += 2
            continue

        # General n't
        if token == "n't":
            normalized.append("not")
            i += 1
            continue

        # =====================================
        # 4. Other contractions
        # =====================================

        contraction_map = {
            "'m": "am",
            "'re": "are",
            "'ve": "have",
            "'ll": "will",
            "'d": "would",
        }

        if token in contraction_map:
            normalized.append(contraction_map[token])
        else:
            normalized.append(token)

        i += 1

    return normalized

In [46]:
test_texts = [
    "The charger doesn't charge the phone",
    "I didn't like it",
    "I can't use this",
    "I'm disappointed",
    "They've sent the wrong case",
    "I won't buy this again",
    "This is terrible :(",
    "Great product :)",
    "The phone's battery is terrible",
    "The case doesn't fit my phone",
]

test_tokens = [
    word_tokenize(normalize_text(text))
    for text in test_texts
]

test_processed = [
    normalize_tokens(tokens)
    for tokens in test_tokens
]

test_result = pd.DataFrame({
    "original": test_texts,
    "tokens_before": test_tokens,
    "tokens_after": test_processed
})

display(test_result)

,original,tokens_before,tokens_after
0,The charger doesn't charge the phone,"[the, charger, does, n't, charge, the, phone]","[the, charger, does, not, charge, the, phone]"
1,I didn't like it,"[i, did, n't, like, it]","[i, did, not, like, it]"
2,I can't use this,"[i, ca, n't, use, this]","[i, can, not, use, this]"
3,I'm disappointed,"[i, 'm, disappointed]","[i, am, disappointed]"
4,They've sent the wrong case,"[they, 've, sent, the, wrong, case]","[they, have, sent, the, wrong, case]"
5,I won't buy this again,"[i, wo, n't, buy, this, again]","[i, will, not, buy, this, again]"
6,This is terrible :(,"[this, is, terrible, :, (]","[this, is, terrible, EMOTION_SAD]"
7,Great product :),"[great, product, :, )]","[great, product, EMOTION_HAPPY]"
8,The phone's battery is terrible,"[the, phone, 's, battery, is, terrible]","[the, phone, 's, battery, is, terrible]"
9,The case doesn't fit my phone,"[the, case, does, n't, fit, my, phone]","[the, case, does, not, fit, my, phone]"


In [47]:
# Apply token normalization to the full corpus

complaints_clean["normalized_tokens"] = (
    complaints_clean["tokens"]
    .map(normalize_tokens)
)

print("Reviews processed:", len(complaints_clean))
print()

display(
    complaints_clean[
        ["normalized_text", "tokens", "normalized_tokens"]
    ].head(10)
)

Reviews processed: 130757



,normalized_text,tokens,normalized_tokens
3,don't care for it. gave it as a gift and they ...,"[do, n't, care, for, it, ., gave, it, as, a, g...","[do, not, care, for, it, ., gave, it, as, a, g..."
5,the product looked exactly like the picture an...,"[the, product, looked, exactly, like, the, pic...","[the, product, looked, exactly, like, the, pic..."
8,do not buy! this item is seriously cheap as he...,"[do, not, buy, !, this, item, is, seriously, c...","[do, not, buy, !, this, item, is, seriously, c..."
12,very cheap broke the first time we put it on :...,"[very, cheap, broke, the, first, time, we, put...","[very, cheap, broke, the, first, time, we, put..."
17,i used this case for not even a week and the b...,"[i, used, this, case, for, not, even, a, week,...","[i, used, this, case, for, not, even, a, week,..."
20,the bow fell off multiple times i had to keep ...,"[the, bow, fell, off, multiple, times, i, had,...","[the, bow, fell, off, multiple, times, i, had,..."
72,i was very excited when i first got this case....,"[i, was, very, excited, when, i, first, got, t...","[i, was, very, excited, when, i, first, got, t..."
75,sent case that didn't fit,"[sent, case, that, did, n't, fit]","[sent, case, that, did, not, fit]"
76,the picture looks waaayyy better than the actu...,"[the, picture, looks, waaayyy, better, than, t...","[the, picture, looks, waaayyy, better, than, t..."
83,i loved the case when i first received it but ...,"[i, loved, the, case, when, i, first, received...","[i, loved, the, case, when, i, first, received..."


In [48]:
# Check reviews containing important semantic patterns

mask = (
    complaints_clean["normalized_text"]
    .str.contains(
        r":\(|:\)|doesn't|didn't|can't|won't",
        case=False,
        regex=True,
        na=False
    )
)

display(
    complaints_clean.loc[
        mask,
        ["reviewText", "normalized_text", "normalized_tokens"]
    ].head(20)
)

,reviewText,normalized_text,normalized_tokens
8,DO NOT BUY! this item is seriously cheap as he...,do not buy! this item is seriously cheap as he...,"[do, not, buy, !, this, item, is, seriously, c..."
12,Very cheap broke the first time we put it on :...,very cheap broke the first time we put it on :...,"[very, cheap, broke, the, first, time, we, put..."
75,Sent case that didn't fit,sent case that didn't fit,"[sent, case, that, did, not, fit]"
92,You are saying that you sell a Universal charg...,you are saying that you sell a universal charg...,"[you, are, saying, that, you, sell, a, univers..."
139,The Samsung car charger has stopped working wi...,the samsung car charger has stopped working wi...,"[the, samsung, car, charger, has, stopped, wor..."
144,Could not power my S2 phone. The LG charger I...,could not power my s2 phone. the lg charger i ...,"[could, not, power, my, s2, phone, ., the, lg,..."
146,It stops working after 3 days and i won't reco...,it stops working after 3 days and i won't reco...,"[it, stops, working, after, 3, days, and, i, w..."
201,"In a nutshell, this is NOT an OEM charger. Al...","in a nutshell, this is not an oem charger. alt...","[in, a, nutshell, ,, this, is, not, an, oem, c..."
207,This charger doesn't charge the battery of my ...,this charger doesn't charge the battery of my ...,"[this, charger, does, not, charge, the, batter..."
220,bought this for my phone and it didn't work. t...,bought this for my phone and it didn't work. t...,"[bought, this, for, my, phone, and, it, did, n..."


In [49]:
from nltk.corpus import stopwords

# NLTK English stopwords
stop_words = set(stopwords.words("english"))

# Giữ lại các từ phủ định vì chúng thay đổi nghĩa của complaint
negation_words = {
    "no",
    "not",
    "never",
    "neither",
    "nor"
}

stop_words = stop_words - negation_words

print("Number of stopwords:", len(stop_words))
print()

print("Negation words kept:")
print(sorted(negation_words))

Number of stopwords: 195

Negation words kept:
['neither', 'never', 'no', 'nor', 'not']


In [50]:
import string

def remove_stopwords(tokens):
    """
    Remove English stopwords and standalone punctuation.

    Keep:
    - negation words: no, not, never, neither, nor
    - semantic emotion tokens
    - normal alphabetic/numeric tokens
    """

    cleaned_tokens = []

    for token in tokens:

        # Giữ semantic emotion tokens
        if token in {"EMOTION_SAD", "EMOTION_HAPPY"}:
            cleaned_tokens.append(token)
            continue

        # Bỏ punctuation đơn lẻ
        if token in string.punctuation:
            continue

        # Bỏ stopwords
        if token in stop_words:
            continue

        cleaned_tokens.append(token)

    return cleaned_tokens

In [51]:
test_stopwords = complaints_clean[
    ["reviewText", "normalized_tokens"]
].head(20).copy()

test_stopwords["tokens_no_stopwords"] = (
    test_stopwords["normalized_tokens"]
    .map(remove_stopwords)
)

display(test_stopwords)

,reviewText,normalized_tokens,tokens_no_stopwords
3,DON'T CARE FOR IT. GAVE IT AS A GIFT AND THEY...,"[do, not, care, for, it, ., gave, it, as, a, g...","[not, care, gave, gift, okay, not, expected]"
5,The product looked exactly like the picture an...,"[the, product, looked, exactly, like, the, pic...","[product, looked, exactly, like, picture, nice..."
8,DO NOT BUY! this item is seriously cheap as he...,"[do, not, buy, !, this, item, is, seriously, c...","[not, buy, item, seriously, cheap, heck, not, ..."
12,Very cheap broke the first time we put it on :...,"[very, cheap, broke, the, first, time, we, put...","[cheap, broke, first, time, put, EMOTION_SAD, ..."
17,I used this case for not even a week and the b...,"[i, used, this, case, for, not, even, a, week,...","[used, case, not, even, week, bow, came, loved..."
20,The bow fell off multiple times I had to keep ...,"[the, bow, fell, off, multiple, times, i, had,...","[bow, fell, multiple, times, keep, gluing, bac..."
72,I was very excited when I first got this case....,"[i, was, very, excited, when, i, first, got, t...","[excited, first, got, case, loved, color, feel..."
75,Sent case that didn't fit,"[sent, case, that, did, not, fit]","[sent, case, not, fit]"
76,The picture looks waaayyy better than the actu...,"[the, picture, looks, waaayyy, better, than, t...","[picture, looks, waaayyy, better, actual, item..."
83,I loved the case when I first received it but ...,"[i, loved, the, case, when, i, first, received...","[loved, case, first, received, shortly, case, ..."


In [52]:
# Download resources needed for POS tagging + lemmatization

resources = [
    ("taggers/averaged_perceptron_tagger_eng",
     "averaged_perceptron_tagger_eng"),
]

for path, resource in resources:
    try:
        nltk.data.find(path)
    except LookupError:
        nltk.download(resource)

In [53]:
from nltk import pos_tag
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer
import string

lemmatizer = WordNetLemmatizer()

SPECIAL_TOKENS = {
    "EMOTION_SAD",
    "EMOTION_HAPPY"
}


def get_wordnet_pos(tag):
    """
    Convert NLTK POS tag to WordNet POS.
    """
    if tag.startswith("J"):
        return wordnet.ADJ
    elif tag.startswith("V"):
        return wordnet.VERB
    elif tag.startswith("N"):
        return wordnet.NOUN
    elif tag.startswith("R"):
        return wordnet.ADV
    else:
        return wordnet.NOUN


def lemmatize_tokens(tokens):
    """
    POS-aware lemmatization.

    IMPORTANT:
    POS tagging is performed BEFORE stopword removal
    so that words such as 'draining' retain their context.
    """

    # Keep special semantic tokens
    normal_tokens = [
        token
        for token in tokens
        if token not in SPECIAL_TOKENS
    ]

    # POS tagging while the original sentence context is intact
    tagged_tokens = pos_tag(normal_tokens)

    lemmatized = []

    for token, tag in tagged_tokens:

        # Skip punctuation here
        if token in string.punctuation:
            continue

        # Keep semantic emotion tokens unchanged
        if token in SPECIAL_TOKENS:
            lemmatized.append(token)
            continue

        pos = get_wordnet_pos(tag)

        lemma = lemmatizer.lemmatize(
            token,
            pos=pos
        )

        lemmatized.append(lemma)

    # Add emotion tokens back in their original occurrences
    for token in tokens:
        if token in SPECIAL_TOKENS:
            lemmatized.append(token)

    return lemmatized

In [54]:
test_sentences = [
    "the battery is draining",
    "the charger is charging",
    "the charger stopped working",
    "the case broke",
    "the screen broke easily",
    "this phone does not work",
]

for sentence in test_sentences:

    tokens = normalize_tokens(
        word_tokenize(
            normalize_text(sentence)
        )
    )

    lemmas = lemmatize_tokens(tokens)

    print(sentence)
    print(f"  tokens  : {tokens}")
    print(f"  lemmas  : {lemmas}")
    print()

the battery is draining
  tokens  : ['the', 'battery', 'is', 'draining']
  lemmas  : ['the', 'battery', 'be', 'drain']

the charger is charging
  tokens  : ['the', 'charger', 'is', 'charging']
  lemmas  : ['the', 'charger', 'be', 'charge']

the charger stopped working
  tokens  : ['the', 'charger', 'stopped', 'working']
  lemmas  : ['the', 'charger', 'stop', 'work']

the case broke
  tokens  : ['the', 'case', 'broke']
  lemmas  : ['the', 'case', 'break']

the screen broke easily
  tokens  : ['the', 'screen', 'broke', 'easily']
  lemmas  : ['the', 'screen', 'break', 'easily']

this phone does not work
  tokens  : ['this', 'phone', 'does', 'not', 'work']
  lemmas  : ['this', 'phone', 'do', 'not', 'work']



In [55]:
from tqdm.auto import tqdm
from nltk import pos_tag_sents
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer
import string

lemmatizer = WordNetLemmatizer()

SPECIAL_TOKENS = {
    "EMOTION_SAD",
    "EMOTION_HAPPY"
}


def get_wordnet_pos(tag):
    if tag.startswith("J"):
        return wordnet.ADJ
    elif tag.startswith("V"):
        return wordnet.VERB
    elif tag.startswith("N"):
        return wordnet.NOUN
    elif tag.startswith("R"):
        return wordnet.ADV
    else:
        return wordnet.NOUN


def lemmatize_batch(token_lists):
    """
    POS-tag and lemmatize a batch of token lists.
    POS tagging happens before stopword removal.
    """

    # POS tagging while sentence context is still intact
    tagged_sentences = pos_tag_sents(token_lists)

    results = []

    for tagged_tokens in tagged_sentences:

        sentence_result = []

        for token, tag in tagged_tokens:

            # Keep semantic emotion tokens
            if token in SPECIAL_TOKENS:
                sentence_result.append(token)
                continue

            # Remove standalone punctuation
            if token in string.punctuation:
                continue

            wn_pos = get_wordnet_pos(tag)

            lemma = lemmatizer.lemmatize(
                token,
                pos=wn_pos
            )

            sentence_result.append(lemma)

        results.append(sentence_result)

    return results


# ==========================================
# Process the full corpus in batches
# ==========================================

token_lists = complaints_clean["normalized_tokens"].tolist()

batch_size = 5000
all_lemmas = []

for start in tqdm(
    range(0, len(token_lists), batch_size),
    desc="Lemmatizing"
):
    batch = token_lists[start:start + batch_size]

    batch_lemmas = lemmatize_batch(batch)

    all_lemmas.extend(batch_lemmas)


complaints_clean["lemmatized_tokens"] = all_lemmas

print("Lemmatization completed.")
print("Reviews processed:", len(complaints_clean))

Lemmatizing: 100%|██████████| 27/27 [04:48<00:00, 10.69s/it]


Lemmatization completed.
Reviews processed: 130757


In [56]:
display(
    complaints_clean[
        [
            "reviewText",
            "normalized_tokens",
            "lemmatized_tokens"
        ]
    ].head(20)
)

,reviewText,normalized_tokens,lemmatized_tokens
3,DON'T CARE FOR IT. GAVE IT AS A GIFT AND THEY...,"[do, not, care, for, it, ., gave, it, as, a, g...","[do, not, care, for, it, give, it, a, a, gift,..."
5,The product looked exactly like the picture an...,"[the, product, looked, exactly, like, the, pic...","[the, product, look, exactly, like, the, pictu..."
8,DO NOT BUY! this item is seriously cheap as he...,"[do, not, buy, !, this, item, is, seriously, c...","[do, not, buy, this, item, be, seriously, chea..."
12,Very cheap broke the first time we put it on :...,"[very, cheap, broke, the, first, time, we, put...","[very, cheap, break, the, first, time, we, put..."
17,I used this case for not even a week and the b...,"[i, used, this, case, for, not, even, a, week,...","[i, use, this, case, for, not, even, a, week, ..."
20,The bow fell off multiple times I had to keep ...,"[the, bow, fell, off, multiple, times, i, had,...","[the, bow, fell, off, multiple, time, i, have,..."
72,I was very excited when I first got this case....,"[i, was, very, excited, when, i, first, got, t...","[i, be, very, excited, when, i, first, get, th..."
75,Sent case that didn't fit,"[sent, case, that, did, not, fit]","[sent, case, that, do, not, fit]"
76,The picture looks waaayyy better than the actu...,"[the, picture, looks, waaayyy, better, than, t...","[the, picture, look, waaayyy, good, than, the,..."
83,I loved the case when I first received it but ...,"[i, loved, the, case, when, i, first, received...","[i, love, the, case, when, i, first, receive, ..."


In [57]:
important_words = [
    "drain",
    "charge",
    "stop",
    "work",
    "break",
    "not"
]

for word in important_words:
    count = complaints_clean["lemmatized_tokens"].apply(
        lambda tokens: word in tokens
    ).sum()

    print(f"{word:10} : {count:,}")

drain      : 794
charge     : 16,006
stop       : 4,916
work       : 26,743
break      : 12,468
not        : 88,141


In [58]:
def remove_stopwords_final(tokens):
    """
    Remove stopwords after lemmatization.

    Keep:
    - negation words: no, not, never, neither, nor
    - semantic emotion tokens
    """

    cleaned_tokens = []

    for token in tokens:

        # Keep semantic emotion tokens
        if token in {"EMOTION_SAD", "EMOTION_HAPPY"}:
            cleaned_tokens.append(token)
            continue

        # Remove stopwords
        if token in stop_words:
            continue

        cleaned_tokens.append(token)

    return cleaned_tokens


complaints_clean["final_tokens"] = (
    complaints_clean["lemmatized_tokens"]
    .map(remove_stopwords_final)
)

print("Final tokenization completed.")
print("Reviews:", len(complaints_clean))

Final tokenization completed.
Reviews: 130757


In [59]:
display(
    complaints_clean[
        [
            "reviewText",
            "lemmatized_tokens",
            "final_tokens"
        ]
    ].head(20)
)

,reviewText,lemmatized_tokens,final_tokens
3,DON'T CARE FOR IT. GAVE IT AS A GIFT AND THEY...,"[do, not, care, for, it, give, it, a, a, gift,...","[not, care, give, gift, okay, not, expect]"
5,The product looked exactly like the picture an...,"[the, product, look, exactly, like, the, pictu...","[product, look, exactly, like, picture, nice, ..."
8,DO NOT BUY! this item is seriously cheap as he...,"[do, not, buy, this, item, be, seriously, chea...","[not, buy, item, seriously, cheap, heck, not, ..."
12,Very cheap broke the first time we put it on :...,"[very, cheap, break, the, first, time, we, put...","[cheap, break, first, time, put, EMOTION_SAD, ..."
17,I used this case for not even a week and the b...,"[i, use, this, case, for, not, even, a, week, ...","[use, case, not, even, week, bow, come, love, ..."
20,The bow fell off multiple times I had to keep ...,"[the, bow, fell, off, multiple, time, i, have,...","[bow, fell, multiple, time, keep, glue, back, ..."
72,I was very excited when I first got this case....,"[i, be, very, excited, when, i, first, get, th...","[excited, first, get, case, love, color, feel,..."
75,Sent case that didn't fit,"[sent, case, that, do, not, fit]","[sent, case, not, fit]"
76,The picture looks waaayyy better than the actu...,"[the, picture, look, waaayyy, good, than, the,...","[picture, look, waaayyy, good, actual, item, h..."
83,I loved the case when I first received it but ...,"[i, love, the, case, when, i, first, receive, ...","[love, case, first, receive, shortly, case, st..."


In [62]:
empty_final_tokens = (
    complaints_clean["final_tokens"]
    .map(len)
    .eq(0)
)

print("Total reviews:", len(complaints_clean))
print("Empty final token lists:", empty_final_tokens.sum())
print(
    "Percentage:",
    round(empty_final_tokens.mean() * 100, 4),
    "%"
)

display(
    complaints_clean.loc[
        empty_final_tokens,
        ["reviewText", "summary", "final_tokens"]
    ].head(30)
)

Total reviews: 130757
Empty final token lists: 19
Percentage: 0.0145 %


,reviewText,summary,final_tokens
5064,"<div id=""video-block-R167MZB0NUVOPD"" class=""a-...","Good Idea, but not in practice",[]
19734,So so,Two Stars,[]
54778,", , . .",One Star,[]
77726,so,One Star,[]
131434,What it is?,One Star,[]
145778,so so,One Star,[]
205053,So so!,One Star,[]
211751,SO SO,Two Stars,[]
288115,S***,One Star,[]
311678,Did,Two Stars,[]


In [63]:
display(
    complaints_clean.loc[
        empty_final_tokens,
        ["reviewText", "summary", "final_tokens"]
    ].head(30)
)

,reviewText,summary,final_tokens
5064,"<div id=""video-block-R167MZB0NUVOPD"" class=""a-...","Good Idea, but not in practice",[]
19734,So so,Two Stars,[]
54778,", , . .",One Star,[]
77726,so,One Star,[]
131434,What it is?,One Star,[]
145778,so so,One Star,[]
205053,So so!,One Star,[]
211751,SO SO,Two Stars,[]
288115,S***,One Star,[]
311678,Did,Two Stars,[]


In [64]:
# Keep only reviews that contain at least one meaningful NLP token

complaints_nlp = complaints_clean[
    complaints_clean["final_tokens"].map(len) > 0
].copy()

print("Before NLP filtering:", len(complaints_clean))
print("After NLP filtering :", len(complaints_nlp))
print("Removed             :", len(complaints_clean) - len(complaints_nlp))

Before NLP filtering: 130757
After NLP filtering : 130738
Removed             : 19


In [65]:
print(
    "Empty final_tokens:",
    complaints_nlp["final_tokens"].map(len).eq(0).sum()
)

print("Shape:", complaints_nlp.shape)

Empty final_tokens: 0
Shape: (130738, 18)


In [66]:
nlp_columns = [
    "asin",
    "reviewText",
    "summary",
    "overall",
    "review_date",
    "year",
    "final_tokens",
]

complaints_nlp = complaints_nlp[nlp_columns].copy()

output_path = (
    project_root
    / "data"
    / "processed"
    / "complaints_nlp.pkl"
)

complaints_nlp.to_pickle(output_path)

print("Saved:", output_path)
print("Shape:", complaints_nlp.shape)

Saved: d:\data mining\e_waste_topic_modeling\data\processed\complaints_nlp.pkl
Shape: (130738, 7)
